In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from itertools import combinations

In [5]:
# =====================================================================
# STEP 1: DATA PREPROCESSING & CLEANING
# =====================================================================

print("Initializing Data Cleaning and Tokenization Pipeline...")

# Load the un-headed transaction dataset
df = pd.read_csv('./dataset/store_data.csv', header=None)
df.head()

Initializing Data Cleaning and Tokenization Pipeline...


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Construct a clean transaction list tracking nested item arrays
# This strips unexpected spaces and ignores empty NaN placeholders
transactions = []
for i in range(len(df)):
    cleaned_basket = [str(df.values[i,j]).strip() for j in range(20) if pd.notna(df.values[i,j])]
    transactions.append(cleaned_basket)

total_transactions = len(transactions)
print(f"Data Cleared Successfully. Tracking {total_transactions} historical transaction baskets.\n")

Data Cleared Successfully. Tracking 7501 historical transaction baskets.



In [7]:
# =====================================================================
# STEP 2: ANALYTICAL PLOT GENERATION
# =====================================================================
print("Generating Data Distribution Charts...")

Generating Data Distribution Charts...


In [8]:
# A. Calculate individual item frequencies for popularity analysis
item_frequencies = Counter()
for basket in transactions:
    item_frequencies.update(basket)

In [9]:
# Isolate top 20 most frequent items (sorted in ascending order for horizontal plotting)
top_20_products = item_frequencies.most_common(20)[::-1]
product_labels = [item[0] for item in top_20_products]
product_counts = [item[1] for item in top_20_products]

# Plot 1: Horizontal Bar Chart of Top Items
plt.barh(product_labels, product_counts, color='skyblue', edgecolor='grey')
plt.title('Top 20 Most Frequent Items Profile')
plt.xlabel('Frequency (Count of Orders)')
plt.ylabel('Product Name')
plt.tight_layout()
plt.savefig('top_items.png')
plt.close()

In [10]:
# B. Calculate transaction size distribution (number of products per basket)
basket_sizes = [len(basket) for basket in transactions]
size_distributions = Counter(basket_sizes)
sorted_sizes = sorted(size_distributions.items())
unique_sizes = [size[0] for size in sorted_sizes]
size_counts = [size[1] for size in sorted_sizes]

In [11]:
# Plot 2: Distribution of Basket Sizes
plt.bar(unique_sizes, size_counts, color='salmon', edgecolor='grey')
plt.title('Distribution of Customer Transaction Basket Sizes')
plt.xlabel('Number of Items Contained in Shopping Cart')
plt.ylabel('Volume of Transactions')
plt.xticks(unique_sizes)
plt.tight_layout()
plt.savefig('transaction_sizes.png')
plt.close()

In [12]:
# =====================================================================
# STEP 3: MACHINE LEARNING (ASSOCIATION RULE LEARNING)
# =====================================================================
print("Initializing Association Rule Learning Engine...")

Initializing Association Rule Learning Engine...


In [13]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

In [14]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

In [15]:
frequent_itemsets = fpgrowth(basket_df, min_support=0.01, use_colnames=True)

In [16]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

In [17]:
rules = rules.sort_values(by="lift", ascending=False)

In [18]:
print("\n================ TOP MARKET BASKET ASSOCIATIONS ================")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).to_string(index=False))
print("================================================================")


================ TOP MARKET BASKET ASSOCIATIONS ================
               antecedents                consequents  support  confidence     lift
           (herb & pepper)              (ground beef) 0.015998    0.323450 3.291994
             (ground beef)            (herb & pepper) 0.015998    0.162822 3.291994
(mineral water, spaghetti)              (ground beef) 0.017064    0.285714 2.907928
             (ground beef) (mineral water, spaghetti) 0.017064    0.173677 2.907928
               (olive oil) (mineral water, spaghetti) 0.010265    0.155870 2.609786
(mineral water, spaghetti)                (olive oil) 0.010265    0.171875 2.609786
                (tomatoes)        (frozen vegetables) 0.016131    0.235867 2.474464
       (frozen vegetables)                 (tomatoes) 0.016131    0.169231 2.474464
                  (shrimp)        (frozen vegetables) 0.016664    0.233209 2.446574
       (frozen vegetables)                   (shrimp) 0.016664    0.174825 2.446574


In [20]:
print("Exporting mined analytical rules to structured storage...")

# Save the full rule repository to a clean spreadsheet file
rules.to_csv('market_basket_rules_report.csv', index=False)

print("Export Complete! 'market_basket_rules_report.csv' has been generated in your project folder.")

Exporting mined analytical rules to structured storage...
Export Complete! 'market_basket_rules_report.csv' has been generated in your project folder.
